## Features

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\josep\iCloudDrive\Projects\BettingEdgeContinued\fantasy\raw_dataset.csv")

# Make sure it's sorted correctly for rolling window calculations
df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

print(df.shape)
df.head()

(34906, 45)


,player_id,player_display_name,position,team,opponent_team,season,week,game_id,fantasy_points,fantasy_points_ppr,...,ffo_expected_pts,ffo_pts_diff,rec_attempt,rush_attempt,rec_yards_gained_exp,rush_yards_gained_exp,rec_touchdown_exp,rush_touchdown_exp,implied_team_total,is_home
0,00-0019596,Tom Brady,QB,TB,NO,2020,1,NaN,20.46,20.46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.25,0
1,00-0019596,Tom Brady,QB,TB,CAR,2020,2,NaN,8.68,8.68,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.75,1
2,00-0019596,Tom Brady,QB,TB,DEN,2020,3,NaN,23.88,23.88,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.25,0
3,00-0019596,Tom Brady,QB,TB,LAC,2020,4,NaN,32.46,32.46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.50,1
4,00-0019596,Tom Brady,QB,TB,CHI,2020,5,NaN,14.12,14.12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.25,0


In [2]:
# Half PPR = standard + (0.5 * receptions)
df["fantasy_points_half_ppr"] = df["fantasy_points"] + (0.5 * df["receptions"])

# Quick sanity check on a known player
jj = df[df["player_display_name"] == "Justin Jefferson"][["season", "week", "receptions", "fantasy_points", "fantasy_points_half_ppr"]].head(10)
print(jj)

       season  week  receptions  fantasy_points  fantasy_points_half_ppr
21018    2020     1           2             2.6                      3.6
21019    2020     2           3             4.4                      5.9
21020    2020     3           7            23.5                     27.0
21021    2020     4           4            10.3                     12.3
21022    2020     5           3             2.3                      3.8
21023    2020     6           9            30.6                     35.1
21024    2020     8           3             2.6                      4.1
21025    2020     9           3             6.4                      7.9
21026    2020    10           8            13.5                     17.5
21027    2020    11           3            14.6                     16.1


In [3]:
# For each player, shift their fantasy points back by 1 row
# So row N contains: features from week W, target = points scored in week W+1
df["target_half_ppr"] = df.groupby("player_id")["fantasy_points_half_ppr"].shift(-1)

# We also need to make sure we don't leak across seasons
# (week 18 of 2021 should NOT predict week 1 of 2022)
df["next_season"] = df.groupby("player_id")["season"].shift(-1)
df.loc[df["next_season"] != df["season"], "target_half_ppr"] = None
df = df.drop(columns=["next_season"])

# These rows (last week of each season per player) will be dropped before training
print(f"Rows with valid target: {df['target_half_ppr'].notna().sum()}")
print(f"Rows without target (last week of season): {df['target_half_ppr'].isna().sum()}")

Rows with valid target: 31290
Rows without target (last week of season): 3616
